In [ ]:
import os
import sys
import time
import signal
import subprocess
from pathlib import Path

# Ensure we are in ai_backend directory
cwd = Path.cwd()
if not (cwd / "main.py").exists() and (cwd / "ai_backend" / "main.py").exists():
    os.chdir(cwd / "ai_backend")

print("Working directory:", Path.cwd())

# Stop previously started processes (if re-running this cell)
for var_name in ["MAIN_PROC", "QWEN_PROC"]:
    proc = globals().get(var_name)
    if proc and proc.poll() is None:
        print(f"Stopping previous {var_name} (PID {proc.pid})...")
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()

python_exe = sys.executable

# Start main.py (AI backend)
MAIN_PROC = subprocess.Popen(
    [python_exe, "main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Start qwen3_tts.py (Qwen TTS backend)
QWEN_PROC = subprocess.Popen(
    [python_exe, "qwen3_tts.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

# Give processes a moment to boot
time.sleep(3)

print(f"main.py started with PID: {MAIN_PROC.pid}, alive={MAIN_PROC.poll() is None}")
print(f"qwen3_tts.py started with PID: {QWEN_PROC.pid}, alive={QWEN_PROC.poll() is None}")

# Print a few startup log lines (non-blocking)
def _drain_lines(proc, name, max_lines=10):
    print(f"\n{name} startup logs:")
    if proc.stdout is None:
        print("(no stdout)")
        return
    for _ in range(max_lines):
        line = proc.stdout.readline()
        if not line:
            break
        print(line.rstrip())

_drain_lines(MAIN_PROC, "main.py")
_drain_lines(QWEN_PROC, "qwen3_tts.py")

print("\nDone. Re-run this cell to restart both services.")